# Notebook for exploatory data analysis and inital rules drafting

## Imports & variable assignment

In [1]:
import pandas as pd
from datetime import datetime, timezone
import glob
import os

In [2]:
opensea_preprocessed_data_folder = "data/processed/opensea/"

## Imports & variable assignment

In [3]:
import pandas as pd
from datetime import datetime, timezone
import glob
import json
import os

In [4]:
opensea_preprocessed_data_folder = "data/processed/opensea/"

In [5]:
def load_parquet_folder(folder_path: str) -> pd.DataFrame:
    """
    Loads and concatenates all Parquet files from a given folder.

    Assumes folder is located relative to repository root (one level above cwd).

    Returns:
        pd.DataFrame: concatenated dataset from all Parquet files
    """

    # Resolve repository root path
    base_dir = os.path.abspath(os.path.join(os.getcwd(), "../.."))
    full_path = os.path.join(base_dir, folder_path)

    # Find parquet files
    parquet_files = glob.glob(os.path.join(full_path, "*.parquet"))

    if not parquet_files:
        print(f"⚠️ No Parquet files found in: {full_path}")
        return pd.DataFrame()

    # Load and concatenate
    dfs = [pd.read_parquet(file) for file in parquet_files]
    df = pd.concat(dfs, ignore_index=True)

    print(f"✅ Loaded {len(df)} rows from {len(parquet_files)} files")

    return df

In [6]:
# Load OpenSea NFT transaction dataset
df_raw = load_parquet_folder(opensea_preprocessed_data_folder)

# Quick preview of dataset structure
df_raw.head()

✅ Loaded 4399 rows from 3 files


,event_timestamp,closing_date,transaction,chain,seller,buyer,nft.name,nft.contract,nft.collection,payment.quantity,payment.symbol,payment.decimals,payment.token_address,token_amount,event_day
0,2025-07-28 14:43:47,1753713827,0x9f763994020190b702b13f63c16448456c225e0c9013...,ethereum,0x7d533ac6cbcee97bf00b7ca10650eab713055736,0x084263a11a1c2998ad4d1e5a79ee57d28ff714c2,CryptoPunk #5418,0xb47e3cd837ddf8e4c57f05d70ab865de6e193bbb,cryptopunks,5.170000e+19,ETH,18,0x0000000000000000000000000000000000000000,51.70,2025-07-28
1,2025-07-28 11:23:11,1753701791,0x3269f0c515ea67d593aed3c6dc50767c660a45ec6e31...,ethereum,0x799224fdb4e635f5833d66e5ba61ffcd1c1fc558,0x978852db1bf809e30576c951733227d2a8fcc07d,CryptoPunk #5512,0xb47e3cd837ddf8e4c57f05d70ab865de6e193bbb,cryptopunks,5.733000e+19,ETH,18,0x0000000000000000000000000000000000000000,57.33,2025-07-28
2,2025-07-28 09:43:23,1753695803,0xed19e54a88fa195c9c75f22d4235a19353358ee37a8d...,ethereum,0x625202e9583039bd61cf599593d269b70521bfa1,0x24fedde1e5e2220256cb1819b7ee7f7b1b20ef5d,CryptoPunk #618,0xb47e3cd837ddf8e4c57f05d70ab865de6e193bbb,cryptopunks,5.149000e+19,ETH,18,0x0000000000000000000000000000000000000000,51.49,2025-07-28
3,2025-07-28 08:47:47,1753692467,0x62cd1d149ad8ffbd9853a0df0b524a6a2bb2eb6cf0c4...,ethereum,0x4fc4457788f12aed1909cea1e4eeafaf2b8a4a00,0x8124ce96fbdce2cd3a80f0ba3b124a98a807be38,CryptoPunk #7364,0xb47e3cd837ddf8e4c57f05d70ab865de6e193bbb,cryptopunks,5.850000e+19,ETH,18,0x0000000000000000000000000000000000000000,58.50,2025-07-28
4,2025-07-28 06:30:47,1753684247,0x342fc7ac6413262273136c4bba0e7a54243b32012253...,ethereum,0x86f39177283138fd6f5e344dfb78675ba4759ada,0x084263a11a1c2998ad4d1e5a79ee57d28ff714c2,CryptoPunk #5512,0xb47e3cd837ddf8e4c57f05d70ab865de6e193bbb,cryptopunks,5.450000e+19,ETH,18,0x0000000000000000000000000000000000000000,54.50,2025-07-28


In [7]:
df_raw.columns

Index(['event_timestamp', 'closing_date', 'transaction', 'chain', 'seller',
       'buyer', 'nft.name', 'nft.contract', 'nft.collection',
       'payment.quantity', 'payment.symbol', 'payment.decimals',
       'payment.token_address', 'token_amount', 'event_day'],
      dtype='str')

In [8]:
df_raw.info()

<class 'pandas.DataFrame'>
RangeIndex: 4399 entries, 0 to 4398
Data columns (total 15 columns):
 #   Column                 Non-Null Count  Dtype         
---  ------                 --------------  -----         
 0   event_timestamp        4399 non-null   datetime64[ms]
 1   closing_date           4399 non-null   int64         
 2   transaction            4399 non-null   str           
 3   chain                  4399 non-null   str           
 4   seller                 4399 non-null   str           
 5   buyer                  4399 non-null   str           
 6   nft.name               4399 non-null   str           
 7   nft.contract           4399 non-null   str           
 8   nft.collection         4399 non-null   str           
 9   payment.quantity       4399 non-null   float64       
 10  payment.symbol         4399 non-null   str           
 11  payment.decimals       4399 non-null   int64         
 12  payment.token_address  4399 non-null   str           
 13  token_amount  

In [9]:
# Create working copy for feature engineering
df = df_raw.copy()

df["event_timestamp"] = pd.to_datetime(df["event_timestamp"], utc=True)

df["event_day"] = df["event_timestamp"].dt.date

df = df.sort_values("event_timestamp").reset_index(drop=True)

# Draft inital Wash Trading rules

## 1. Initial self-trades check

In [10]:
# --------------------------------------------------
# 1. Self-trade detection
# --------------------------------------------------
# Definition: self-trade occurs when seller and buyer are the same wallet.
# This is a direct anomaly signal in NFT wash trading detection.

df["self_trade"] = df["seller"] == df["buyer"]

self_trade_counts = df["self_trade"].value_counts()

print("Self-trade distribution:")
print(self_trade_counts)

Self-trade distribution:
self_trade
False    4391
True        8
Name: count, dtype: int64


In [11]:
# --------------------------------------------------
# Self-trade inspection
# --------------------------------------------------
# Definition: self-trades occur when seller and buyer wallets are identical.

self_trades = df[df["self_trade"] == True]

if self_trades.empty:
    print("No self-trades detected.")
else:
    print(f" 🚨 Detected {len(self_trades)} self-trades")

    display(
        self_trades[
            [
                "event_timestamp",
                "transaction",
                "seller",
                "buyer",
                "nft.name",
                "token_amount",
                "nft.collection",
            ]
        ].head(20)
    )

 🚨 Detected 8 self-trades


,event_timestamp,transaction,seller,buyer,nft.name,token_amount,nft.collection
49,2025-07-02 16:41:11+00:00,0xc85422fd4415de934241e2fd3179eb91dfc5da699ddc...,0xb751379ff5dbc7ae98279486f151172dd37abaa3,0xb751379ff5dbc7ae98279486f151172dd37abaa3,#4049,11.19375,boredapeyachtclub
1307,2025-07-21 16:23:35+00:00,0x4a47f805d07a6bbe91683c65760d1707bf1c44fb7a89...,0x54d28b26487e6529cf447434bccf439bea352e5d,0x54d28b26487e6529cf447434bccf439bea352e5d,#1688,12.83550,boredapeyachtclub
2097,2025-08-02 13:48:11+00:00,0xc8b673ea6dc7fa237f1eed2f2b7396f573a6ac2b550b...,0x54d28b26487e6529cf447434bccf439bea352e5d,0x54d28b26487e6529cf447434bccf439bea352e5d,#640,12.08925,boredapeyachtclub
2488,2025-08-10 01:05:11+00:00,0xa4aea2116accd60f3f14846cba9aabe5eaa7a3953db4...,0x54d28b26487e6529cf447434bccf439bea352e5d,0x54d28b26487e6529cf447434bccf439bea352e5d,Pudgy Penguin #7077,15.92000,pudgypenguins
2713,2025-08-14 13:36:23+00:00,0x3d970aa61501f3f9d19e2e36f5cd434b4d2542a90881...,0x54d28b26487e6529cf447434bccf439bea352e5d,0x54d28b26487e6529cf447434bccf439bea352e5d,Pudgy Penguin #3681,13.43250,pudgypenguins
2917,2025-08-18 12:11:35+00:00,0xfc35f2a18ab426d3784de61bb90c57cd870e2db3abcb...,0x54d28b26487e6529cf447434bccf439bea352e5d,0x54d28b26487e6529cf447434bccf439bea352e5d,#5457,12.62655,boredapeyachtclub
3741,2025-09-11 18:25:59+00:00,0x09ecd6a6f589843389f526e46be3d87ee2019cb9c8e7...,0x54d28b26487e6529cf447434bccf439bea352e5d,0x54d28b26487e6529cf447434bccf439bea352e5d,#9494,9.20375,boredapeyachtclub
4366,2025-09-29 02:57:59+00:00,0xe95fb67ee5dee29935c092b3ff34ba139297a77d8af7...,0x3caebdddfb38d801cf4a2ec638085c36abea9c9e,0x3caebdddfb38d801cf4a2ec638085c36abea9c9e,Pudgy Penguin #1915,10.15740,pudgypenguins


## 2. Holding period

In [12]:
# --------------------------------------------------
# 2. Holding period analysis
# --------------------------------------------------
# Definition: holding period measures time difference between consecutive sales
# of the same NFT. Very short holding periods may indicate wash trading.

df["prev_sale_time"] = df.groupby("nft.name")["event_timestamp"].shift(1)

df["holding_seconds"] = (
    df["event_timestamp"] - df["prev_sale_time"]
).dt.total_seconds()

In [13]:
# Thresholds (based on heuristic assumptions from literature / domain intuition)
ONE_DAY = 86_400   # 24 hours
ONE_HOUR = 3_600   # 1 hour

long_holds = df[df["holding_seconds"] > ONE_DAY]

print(f" 🚨 Long holding periods (>24h): {len(long_holds)}")

 🚨 Long holding periods (>24h): 1380


In [14]:
short_holds = df[df["holding_seconds"] <= ONE_HOUR]

print(f" 🚨 Short holding periods (<=1h): {len(short_holds)}")

display(
    short_holds[
        [
            "event_timestamp",
            "prev_sale_time",
            "holding_seconds",
            "seller",
            "buyer",
            "nft.name",
            "token_amount",
        ]
    ].head(20)
)

 🚨 Short holding periods (<=1h): 381


,event_timestamp,prev_sale_time,holding_seconds,seller,buyer,nft.name,token_amount
6,2025-07-01 13:51:47+00:00,2025-07-01 13:45:47+00:00,360.0,0x94e9d73a2aeec807d85756b84759965b41866450,0x621c70de47c35be4622c891555a6016cde2e1a61,Pudgy Penguin #2897,9.598956
18,2025-07-01 21:57:47+00:00,2025-07-01 21:57:47+00:00,0.0,0x5129750f8c36b794e564d3fc5ef01c283d79ec36,0xfc02d4571ba182ac2fdf3e14fcb37f32e79bd28c,#7246,20.000000
48,2025-07-02 16:41:11+00:00,2025-07-02 16:41:11+00:00,0.0,0x7e7022f8879d88bcc5d288b229737adb4b1f39cb,0xb751379ff5dbc7ae98279486f151172dd37abaa3,#4049,10.630000
49,2025-07-02 16:41:11+00:00,2025-07-02 16:41:11+00:00,0.0,0xb751379ff5dbc7ae98279486f151172dd37abaa3,0xb751379ff5dbc7ae98279486f151172dd37abaa3,#4049,11.193750
68,2025-07-03 15:49:47+00:00,2025-07-03 15:39:59+00:00,588.0,0x8bab603fde4824950fa81268a84ef9cbb89cad69,0x843fcb0cf448e714e735eb4b30e81186b9d58ec6,#524,10.740000
78,2025-07-03 19:30:11+00:00,2025-07-03 18:40:47+00:00,2964.0,0xa6a8c4aff47e1a2e907cd1ef5161f14db7e0d9ba,0xf76246b0842c92ad5bd745973ca9eb85b937b126,#9129,10.430000
97,2025-07-04 16:31:59+00:00,2025-07-04 16:30:11+00:00,108.0,0x1fee3385b22d69e93209db2042be58fcac57b59b,0xf2a7be2d334db950b8e490def5412827eac8494c,Pudgy Penguin #5156,9.300000
117,2025-07-07 12:01:11+00:00,2025-07-07 12:00:23+00:00,48.0,0x8bab603fde4824950fa81268a84ef9cbb89cad69,0xf76246b0842c92ad5bd745973ca9eb85b937b126,#3465,11.540000
118,2025-07-07 12:01:59+00:00,2025-07-07 12:01:11+00:00,48.0,0xf76246b0842c92ad5bd745973ca9eb85b937b126,0x66816bd969ff35688e9fd82b3e46fd505260216a,#3465,11.580000
132,2025-07-07 23:20:59+00:00,2025-07-07 22:21:59+00:00,3540.0,0x2463a7b933a50188ff83ab5c703fe6ef3a0449c5,0xf76246b0842c92ad5bd745973ca9eb85b937b126,Pudgy Penguin #936,8.910000


## 3. Initial NFT flip count

In [15]:
# --------------------------------------------------
# 3. NFT flip frequency
# --------------------------------------------------
# Definition: number of transactions per NFT.
# High frequency may indicate artificial trading activity.

df["nft_id"] = df["nft.contract"].astype(str) + "_" + df["nft.name"].astype(str)

flip_counts = df.groupby("nft_id").size()
df["nft_flip_count"] = df["nft_id"].map(flip_counts)

flip_distribution = df["nft_flip_count"].value_counts().sort_index()

print("NFT flip frequency distribution:")
print(flip_distribution)

NFT flip frequency distribution:
nft_flip_count
1     869
2     970
3     693
4     548
5     340
6     348
7     112
8     112
9      99
10     70
11     22
12     24
13     26
14     28
17     17
22     22
99     99
Name: count, dtype: int64


## 4. Wallet-pair behavior (VERY important)
Same pair trades multiple times
Same NFT traded back and forth between same pair

In [16]:
# --------------------------------------------------
# 4. Wallet pair interaction analysis
# --------------------------------------------------
# Definition: measures how often the same wallet pair trades with each other.
# High repetition may indicate coordinated trading behavior.

pair_counts = (
    df.groupby(["seller", "buyer"])
      .size()
      .reset_index(name="pair_trade_count")
)

df = df.merge(pair_counts, on=["seller", "buyer"], how="left")

REPEAT_THRESHOLD = 5

repeated_pairs = df[df["pair_trade_count"] >= REPEAT_THRESHOLD]

print(f" 🚨 High-frequency wallet pairs (>= {REPEAT_THRESHOLD} trades): {len(repeated_pairs)}")

 🚨 High-frequency wallet pairs (>= 5 trades): 76


## Circular trading detection (A → B → A)

In [17]:
# --------------------------------------------------
# Circular trading detection (improved heuristic)
# --------------------------------------------------
# Goal:
# Detect repeated trading cycles between wallets for the same NFT.
# This captures more realistic wash trading behavior than simple A→B→A patterns.

# Step 1: Create directed trading edges per NFT
df = df.sort_values(["nft_id", "event_timestamp"])

# Step 2: Track previous owner for each NFT
df["prev_owner"] = df.groupby("nft_id")["seller"].shift(1)

# Step 3: Detect immediate reversals (A → B → A)
df["immediate_round_trip"] = df["buyer"] == df["prev_owner"]

# Step 4: Detect repeated interactions between same wallet pairs
# (A↔B occurring multiple times for same NFT)
pair_cycle_counts = (
    df.groupby(["nft_id", "seller", "buyer"])
      .size()
      .reset_index(name="pair_repeat_count")
)

df = df.merge(pair_cycle_counts, on=["nft_id", "seller", "buyer"], how="left")

df["repeated_pair_cycle"] = df["pair_repeat_count"] >= 2

# Step 5: Combined circular trading signal
df["circular_trading"] = (
    df["immediate_round_trip"] | df["repeated_pair_cycle"]
)

# Step 6: Extract suspicious circular trades
circular_trades = df[df["circular_trading"]]

print(f"🚨 Detected {len(circular_trades)} circular trading-related transactions")

display(
    circular_trades[
        [
            "event_timestamp",
            "nft.name",
            "seller",
            "buyer",
            "nft_id",
            "immediate_round_trip",
            "pair_repeat_count",
            "token_amount",
        ]
    ].head(20)
)

🚨 Detected 64 circular trading-related transactions


,event_timestamp,nft.name,seller,buyer,nft_id,immediate_round_trip,pair_repeat_count,token_amount
32,2025-09-24 22:05:11+00:00,CryptoPunk #1613,0x1919db36ca2fa2e15f9000fd9cdc2edcf863e685,0x80bf7db69556d9521c03461978b8fc731dbbd4e4,0xb47e3cd837ddf8e4c57f05d70ab865de6e193bbb_Cry...,True,1,4.649000e+01
36,2025-07-28 19:36:47+00:00,CryptoPunk #1668,0x1919db36ca2fa2e15f9000fd9cdc2edcf863e685,0x7650f20becf1d3d37407e281645c638d1ae8939c,0xb47e3cd837ddf8e4c57f05d70ab865de6e193bbb_Cry...,False,2,5.389000e+01
37,2025-07-28 19:36:59+00:00,CryptoPunk #1668,0x1919db36ca2fa2e15f9000fd9cdc2edcf863e685,0x7650f20becf1d3d37407e281645c638d1ae8939c,0xb47e3cd837ddf8e4c57f05d70ab865de6e193bbb_Cry...,False,2,5.389000e+01
101,2025-09-17 21:04:23+00:00,CryptoPunk #267,0xea889aa7ede714a4cff37e81487a3919c28f2e64,0x8b971c66d743ab519d273b2b36de15814f584369,0xb47e3cd837ddf8e4c57f05d70ab865de6e193bbb_Cry...,True,1,4.749000e+01
111,2025-09-08 18:38:35+00:00,CryptoPunk #2783,0xc50673edb3a7b94e8cad8a7d4e0cd68864e33edf,0xd5ef7d3d225770bcfc4a46f9cef413f440610dee,0xb47e3cd837ddf8e4c57f05d70ab865de6e193bbb_Cry...,True,1,1.200000e-16
153,2025-09-28 10:30:35+00:00,CryptoPunk #3483,0x3042887f97821ec36be64d2677efd2e943a4cb0f,0x6ec1b656f9ea50c89827c7e820c303a6039550e3,0xb47e3cd837ddf8e4c57f05d70ab865de6e193bbb_Cry...,True,1,4.495000e+01
279,2025-09-04 01:31:23+00:00,CryptoPunk #5295,0x73f69db86dd39ac8172723d7a150a6aeb316d08a,0x32aa011c1e2ce7aab918e97f5567b871709e654b,0xb47e3cd837ddf8e4c57f05d70ab865de6e193bbb_Cry...,False,2,4.900000e+01
280,2025-09-04 01:31:47+00:00,CryptoPunk #5295,0x73f69db86dd39ac8172723d7a150a6aeb316d08a,0x32aa011c1e2ce7aab918e97f5567b871709e654b,0xb47e3cd837ddf8e4c57f05d70ab865de6e193bbb_Cry...,False,2,4.900000e+01
295,2025-09-23 18:24:23+00:00,CryptoPunk #5467,0x1919db36ca2fa2e15f9000fd9cdc2edcf863e685,0x00000f91109c4d0007e90000d9facad5298a0cac,0xb47e3cd837ddf8e4c57f05d70ab865de6e193bbb_Cry...,True,1,4.557000e+01
335,2025-09-27 15:45:35+00:00,CryptoPunk #6058,0x0cc3b5015c678685f3552fb1a1bfa53112b71486,0x1919db36ca2fa2e15f9000fd9cdc2edcf863e685,0xb47e3cd837ddf8e4c57f05d70ab865de6e193bbb_Cry...,True,1,4.500000e+01


## 5. Price abnormality (statistical signal)

In [18]:
# --------------------------------------------------
# Price anomaly detection (robust statistical method)
# --------------------------------------------------
# Goal:
# Identify abnormal NFT transaction prices relative to
# their collection-day distribution using robust statistics
# (median + MAD-based z-score).

# Step 1: Compute daily collection statistics
daily_stats = (
    df.groupby(["nft.collection", "event_day"])
      .agg(
          median_price=("token_amount", "median"),
          mad_price=("token_amount", lambda x: (x - x.median()).abs().median()),
          n_trades=("token_amount", "count")
      )
      .reset_index()
)

# Step 2: Keep only statistically reliable groups
# (avoid unstable estimates for very small samples)
daily_stats = daily_stats[daily_stats["n_trades"] >= 5]

# Step 3: Merge statistics back into main dataset
df = df.merge(
    daily_stats,
    on=["nft.collection", "event_day"],
    how="left"
)

# Step 4: Prevent division by zero / numerical instability
df["mad_price"] = df["mad_price"].replace(0, 1e-9)

# Step 5: Compute robust z-score (based on MAD)
df["price_zscore"] = (
    0.6745 *
    (df["token_amount"] - df["median_price"]) /
    df["mad_price"]
)

# Step 6: Flag abnormal prices
PRICE_ZSCORE_THRESHOLD = 3.5
df["price_abnormality"] = df["price_zscore"].abs() > PRICE_ZSCORE_THRESHOLD

# Step 7: Extract anomalies for inspection
abnormal_prices = df[df["price_abnormality"]]

print(f"🚨 Detected {len(abnormal_prices)} price anomalies")

display(
    abnormal_prices[
        [
            "event_timestamp",
            "nft.name",
            "seller",
            "buyer",
            "token_amount",
            "median_price",
            "price_zscore",
        ]
    ].head(20)
)

🚨 Detected 313 price anomalies


,event_timestamp,nft.name,seller,buyer,token_amount,median_price,price_zscore
1,2025-08-05 16:41:47+00:00,CryptoPunk #1021,0x2f2f237d2e655cc0a6f6fef761e5aef13087e71f,0x0232d1083e970f0c78f56202b9a666b526fa379f,720.000000,58.000,71.443040
4,2025-08-17 15:49:11+00:00,CryptoPunk #1032,0x198eff321af039138ccef8045af4154c20ae6302,0x1919db36ca2fa2e15f9000fd9cdc2edcf863e685,52.000000,49.990,15.063833
6,2025-07-22 07:04:35+00:00,CryptoPunk #1068,0xaf184b4cbc73a9ca2f51c4a4d80ed67a2578e9f4,0x6b92686c40747c85809a6772d0eda8e22a77c60c,56.000000,50.000,4.047000
7,2025-08-18 22:41:11+00:00,CryptoPunk #1082,0xf65c1c42745606e397bb2993e25e09382a16fb87,0x089f31c6cc14eff1229dd56d015c59cb3c7e6c5a,80.000000,52.000,9.396020
17,2025-09-28 04:06:59+00:00,CryptoPunk #1350,0x6ec1b656f9ea50c89827c7e820c303a6039550e3,0x1244eae9fa2c064453b5f605d708c0a0bfba4838,46.291029,45.000,17.415977
25,2025-07-25 06:14:59+00:00,CryptoPunk #1484,0x0233a4970e5fa014f263b4a52427e997291e6cca,0x8000df0d497b9bebb1256d08a46828119b4d6420,148.500000,55.000,12.613150
41,2025-08-15 20:01:11+00:00,CryptoPunk #1714,0x0232d1083e970f0c78f56202b9a666b526fa379f,0x0233a4970e5fa014f263b4a52427e997291e6cca,77.000000,53.095,5.209668
62,2025-07-25 06:14:47+00:00,CryptoPunk #1946,0xf5a4ba515dd36a42e9634a3ac1f1c58fa24d3acd,0x8000df0d497b9bebb1256d08a46828119b4d6420,145.000000,55.000,12.141000
67,2025-07-23 12:00:35+00:00,CryptoPunk #2041,0xaf184b4cbc73a9ca2f51c4a4d80ed67a2578e9f4,0x6f32fea0503b679a61ec14dbdb36c48ce7d8ef3b,59.000000,49.500,4.271833
74,2025-07-28 02:18:11+00:00,CryptoPunk #2165,0xa56268c56983fd7f3892a3677ced8c1b27750dd1,0x8000df0d497b9bebb1256d08a46828119b4d6420,69.000000,55.250,3.573940


## 6. Final wash trading score

In [19]:
# --------------------------------------------------
# Wash trading risk scoring model
# --------------------------------------------------
# This section computes a heuristic risk score for each transaction
# based on behavioral, temporal, network, and price-based signals.
# The weights are manually defined (expert-driven heuristic approach).

# Step 1: Compute composite wash trading score
df["wash_score"] = (
    df["self_trade"].fillna(False).astype(int) * 35 +
    df["immediate_round_trip"].fillna(False).astype(int) * 25 +
    (df["pair_trade_count"].fillna(0) >= 5).astype(int) * 15 +
    (df["holding_seconds"].fillna(0) <= 3600).astype(int) * 10 +
    (df["nft_flip_count"].fillna(0) >= 3).astype(int) * 10 +
    df["price_abnormality"].fillna(False).astype(int) * 5
)

# Step 2: Normalize score range
df["wash_score"] = df["wash_score"].clip(0, 100)

# Step 3: Risk classification function
def classify_score(score):
    if score >= 70:
        return "High Risk"
    elif score >= 40:
        return "Medium Risk"
    else:
        return "Low Risk"

df["wash_risk"] = df["wash_score"].apply(classify_score)

# Step 4: Extract suspicious transactions
suspicious_transactions = df[df["wash_score"] >= 40]

print(f"🚨 Detected {len(suspicious_transactions)} suspicious transactions")

display(
    suspicious_transactions[
        [
            "event_timestamp",
            "nft.name",
            "seller",
            "buyer",
            "token_amount",
            "wash_score",
            "wash_risk",
            "self_trade",
            "immediate_round_trip",
            "pair_trade_count",
            "nft_flip_count",
            "price_zscore",
        ]
    ].head(30)
)

🚨 Detected 18 suspicious transactions


,event_timestamp,nft.name,seller,buyer,token_amount,wash_score,wash_risk,self_trade,immediate_round_trip,pair_trade_count,nft_flip_count,price_zscore
32,2025-09-24 22:05:11+00:00,CryptoPunk #1613,0x1919db36ca2fa2e15f9000fd9cdc2edcf863e685,0x80bf7db69556d9521c03461978b8fc731dbbd4e4,46.490000,45,Medium Risk,False,True,2,3,-1.332209
101,2025-09-17 21:04:23+00:00,CryptoPunk #267,0xea889aa7ede714a4cff37e81487a3919c28f2e64,0x8b971c66d743ab519d273b2b36de15814f584369,47.490000,45,Medium Risk,False,True,1,4,NaN
407,2025-08-14 19:16:47+00:00,CryptoPunk #6741,0x1919db36ca2fa2e15f9000fd9cdc2edcf863e685,0xf4c40bf7070fdcf64ecf020bcb583738a6cc3bcd,50.000000,45,Medium Risk,False,True,3,5,0.004003
632,2025-09-27 16:27:59+00:00,CryptoPunk #9763,0x1919db36ca2fa2e15f9000fd9cdc2edcf863e685,0x80bf7db69556d9521c03461978b8fc731dbbd4e4,44.800000,45,Medium Risk,False,True,2,3,-0.009299
824,2025-07-21 16:23:35+00:00,#1688,0x54d28b26487e6529cf447434bccf439bea352e5d,0x54d28b26487e6529cf447434bccf439bea352e5d,12.835500,70,High Risk,True,False,6,7,0.218150
1020,2025-07-25 06:08:11+00:00,#300,0x3e6e8b955e6f5484fc95200737e4d092b35c577c,0xf76246b0842c92ad5bd745973ca9eb85b937b126,12.100000,45,Medium Risk,False,True,1,6,0.367909
1122,2025-07-02 16:41:11+00:00,#4049,0x7e7022f8879d88bcc5d288b229737adb4b1f39cb,0xb751379ff5dbc7ae98279486f151172dd37abaa3,10.630000,45,Medium Risk,False,True,1,7,0.049354
1123,2025-07-02 16:41:11+00:00,#4049,0xb751379ff5dbc7ae98279486f151172dd37abaa3,0xb751379ff5dbc7ae98279486f151172dd37abaa3,11.193750,55,Medium Risk,True,False,1,7,1.904229
1265,2025-08-14 19:39:11+00:00,#4795,0x1ea27bce786a81022dfc156059771e8d3279a9a6,0xc67268421283cce9da893ae59767ad8571b558e7,12.159799,45,Medium Risk,False,True,2,13,0.166520
1353,2025-08-18 12:11:35+00:00,#5457,0x54d28b26487e6529cf447434bccf439bea352e5d,0x54d28b26487e6529cf447434bccf439bea352e5d,12.626550,95,High Risk,True,True,6,6,0.789760


In [20]:
suspicious_rate = len(suspicious_transactions) / len(df) * 100
print(suspicious_rate)

0.40918390543305294


In [21]:
df["wash_risk"].value_counts()

wash_risk
Low Risk       4381
Medium Risk      12
High Risk         6
Name: count, dtype: int64

In [22]:
display(suspicious_transactions)

,event_timestamp,closing_date,transaction,chain,seller,buyer,nft.name,nft.contract,nft.collection,payment.quantity,...,pair_repeat_count,repeated_pair_cycle,circular_trading,median_price,mad_price,n_trades,price_zscore,price_abnormality,wash_score,wash_risk
32,2025-09-24 22:05:11+00:00,1758751511,0x098be511a3af28c7d0a2e6f68e0b3c39be9e29e86cd2...,ethereum,0x1919db36ca2fa2e15f9000fd9cdc2edcf863e685,0x80bf7db69556d9521c03461978b8fc731dbbd4e4,CryptoPunk #1613,0xb47e3cd837ddf8e4c57f05d70ab865de6e193bbb,cryptopunks,4.649000e+19,...,1,False,True,48.000000,0.764516,5.0,-1.332209,False,45,Medium Risk
101,2025-09-17 21:04:23+00:00,1758143063,0x2a046801e7b5c5a12a4736939ee56e4c2d162e2eca10...,ethereum,0xea889aa7ede714a4cff37e81487a3919c28f2e64,0x8b971c66d743ab519d273b2b36de15814f584369,CryptoPunk #267,0xb47e3cd837ddf8e4c57f05d70ab865de6e193bbb,cryptopunks,4.749000e+19,...,1,False,True,NaN,NaN,NaN,NaN,False,45,Medium Risk
407,2025-08-14 19:16:47+00:00,1755199007,0xe05d4d5e7a7612a839b06575960306770ca082eeddd2...,ethereum,0x1919db36ca2fa2e15f9000fd9cdc2edcf863e685,0xf4c40bf7070fdcf64ecf020bcb583738a6cc3bcd,CryptoPunk #6741,0xb47e3cd837ddf8e4c57f05d70ab865de6e193bbb,cryptopunks,5.000000e+19,...,2,True,True,49.990000,1.685000,12.0,0.004003,False,45,Medium Risk
632,2025-09-27 16:27:59+00:00,1758990479,0xe45a908f876fe4d7188dffb258b3c13ccf9d49daa664...,ethereum,0x1919db36ca2fa2e15f9000fd9cdc2edcf863e685,0x80bf7db69556d9521c03461978b8fc731dbbd4e4,CryptoPunk #9763,0xb47e3cd837ddf8e4c57f05d70ab865de6e193bbb,cryptopunks,4.480000e+19,...,1,False,True,44.802549,0.184925,6.0,-0.009299,False,45,Medium Risk
824,2025-07-21 16:23:35+00:00,1753115015,0x4a47f805d07a6bbe91683c65760d1707bf1c44fb7a89...,ethereum,0x54d28b26487e6529cf447434bccf439bea352e5d,0x54d28b26487e6529cf447434bccf439bea352e5d,#1688,0xbc4ca0eda7647a8ab7c2061c2e118a18a936f13d,boredapeyachtclub,1.283550e+19,...,1,False,False,12.699500,0.420500,72.0,0.218150,False,70,High Risk
1020,2025-07-25 06:08:11+00:00,1753423691,0xc848f3eab84107165ce77985a9fdbf11bd40f66e9be4...,ethereum,0x3e6e8b955e6f5484fc95200737e4d092b35c577c,0xf76246b0842c92ad5bd745973ca9eb85b937b126,#300,0xbc4ca0eda7647a8ab7c2061c2e118a18a936f13d,boredapeyachtclub,1.210000e+19,...,1,False,True,12.040000,0.110000,7.0,0.367909,False,45,Medium Risk
1122,2025-07-02 16:41:11+00:00,1751474471,0xc85422fd4415de934241e2fd3179eb91dfc5da699ddc...,ethereum,0x7e7022f8879d88bcc5d288b229737adb4b1f39cb,0xb751379ff5dbc7ae98279486f151172dd37abaa3,#4049,0xbc4ca0eda7647a8ab7c2061c2e118a18a936f13d,boredapeyachtclub,1.063000e+19,...,1,False,True,10.615000,0.205000,16.0,0.049354,False,45,Medium Risk
1123,2025-07-02 16:41:11+00:00,1751474471,0xc85422fd4415de934241e2fd3179eb91dfc5da699ddc...,ethereum,0xb751379ff5dbc7ae98279486f151172dd37abaa3,0xb751379ff5dbc7ae98279486f151172dd37abaa3,#4049,0xbc4ca0eda7647a8ab7c2061c2e118a18a936f13d,boredapeyachtclub,1.119375e+19,...,1,False,False,10.615000,0.205000,16.0,1.904229,False,55,Medium Risk
1265,2025-08-14 19:39:11+00:00,1755200351,0xfcc11d9c3344f9a8bbebc48fb081ab85ff8b6ac14e07...,ethereum,0x1ea27bce786a81022dfc156059771e8d3279a9a6,0xc67268421283cce9da893ae59767ad8571b558e7,#4795,0xbc4ca0eda7647a8ab7c2061c2e118a18a936f13d,boredapeyachtclub,1.215980e+19,...,1,False,True,12.094500,0.264500,94.0,0.166520,False,45,Medium Risk
1353,2025-08-18 12:11:35+00:00,1755519095,0xfc35f2a18ab426d3784de61bb90c57cd870e2db3abcb...,ethereum,0x54d28b26487e6529cf447434bccf439bea352e5d,0x54d28b26487e6529cf447434bccf439bea352e5d,#5457,0xbc4ca0eda7647a8ab7c2061c2e118a18a936f13d,boredapeyachtclub,1.262655e+19,...,1,False,True,11.797800,0.707800,21.0,0.789760,False,95,High Risk


In [24]:
suspicious_transactions["nft.collection"].value_counts()

nft.collection
boredapeyachtclub    10
cryptopunks           4
pudgypenguins         4
Name: count, dtype: int64